# ChEMBL fragment space analysis
Henrik Bermanis 09/10/2026

## Imports and settings

In [ ]:
import csv
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator

# ChEMBL trips loads of parser warnings in the passes below.
RDLogger.DisableLog("rdApp.*")

DB = Path("Data/chembl_37/chembl_37_sqlite/chembl_37.db")
FRAGMENTS = Path("Data/chembl_37_fragments.csv")
EMBEDDING = Path("Data/chembl_37_fragments_umap.csv")
FIGURE_STEM = Path("Artifacts/chembl_37_fragments_umap")

# Just to make sure memory is more or less constant 
BATCH = 50000

# Same consts as in the old script
RADIUS = 2          
N_BITS = 2048
N_NEIGHBORS = 15
MIN_DIST = 0.1
N_JOBS = 24 # All Studio's power cores

print(DB, f"{DB.stat().st_size / 1e9:.1f} GB" if DB.exists() else "MISSING")

# Extract the fragments

Use SQL against ChEMBL mol properties so only the frags reach rdkit and the massive file is never loaded in all. 
<br><br>
Change the query to get different subsets of frags. NOTE: MW is judged by the freebase weight to accomodate salts. 

In [ ]:
QUERY = """
    SELECT md.chembl_id, cs.canonical_smiles, cp.mw_freebase
    FROM compound_structures AS cs
    JOIN compound_properties AS cp USING (molregno)
    JOIN molecule_dictionary AS md USING (molregno)
    WHERE cs.canonical_smiles IS NOT NULL
      AND cp.mw_freebase <= 300
      AND cp.alogp <= 100
      AND cp.hbd <= 100
      AND cp.hba <= 100
      AND cp.rtb <= 100
"""

# Read-only URI, so nothing can modify the 30 GB release.
connection = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)

# Return frag proportion
n_smiles = connection.execute("SELECT COUNT(*) FROM compound_structures WHERE canonical_smiles IS NOT NULL").fetchone()[0]
n_frags = connection.execute(f"SELECT COUNT(*) FROM ({QUERY})").fetchone()[0]

print(f"structures with SMILES: {n_smiles:,}")
print(f"rule-of-three rows:     {n_frags:,} ({100 * n_frags / n_smiles:.1f}%)")

## Clean mols, stream the release, and write the frags file

Parse, drop the counter-ion, canonicalise. Batched fetch, streaming write, dedup on canonical SMILES.

In [ ]:
def clean(smiles, mw_freebase):
    """
    Parse, drop the counter-ion if present, and return a canonical smiles. 
    Keeps the freebase frag to avoid instances where counter-ion is heavier
    and thus kept, e.g., pamoates, tosylates, picrates, etc.  
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    if "." in smiles:
        pieces = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)
        mol = min(pieces, key=lambda piece: abs(Descriptors.MolWt(piece) - mw_freebase))
    return Chem.MolToSmiles(mol)

cursor = connection.execute(QUERY)
seen = set()
n_rows = 0

FRAGMENTS.parent.mkdir(parents=True, exist_ok=True)
with FRAGMENTS.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["chembl_id", "smiles"])

    while rows := cursor.fetchmany(BATCH):
        n_rows += len(rows)
        
        for chembl_id, smiles, mw_freebase in rows:
            canonical = clean(smiles, mw_freebase)
            if canonical and canonical not in seen:
                seen.add(canonical)
                writer.writerow([chembl_id, canonical])
        print(f"  {n_rows:,} rows scanned, {len(seen):,} fragments kept", flush=True)

connection.close()
print(f"\n{n_rows:,} rows -> {len(seen):,} unique fragments -> {FRAGMENTS}")

## Quick check of the frags. 

A grid of random structures to visualise.

In [ ]:
from rdkit.Chem import Draw

fragments = pd.read_csv(FRAGMENTS)
print(f"{len(fragments):,} rows | {fragments['smiles'].nunique():,} unique SMILES")

sample = fragments["smiles"].sample(12, random_state=0).tolist()
Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in sample], molsPerRow=4, subImgSize=(220, 180), legends=sample,)

# Filter Diagnostics

What's filtered out and what's kep. Read-only and adjacent to the workflow so nothing here feeds the fprints below.
Every check prints a summary and keeps its dataframe in `DIAGNOSTIC_FRAMES`, all of which get written to `Data/diagnostics/` in the last cell.

In [ ]:
import re
from rdkit.Chem import Crippen, Lipinski

DIAGNOSTICS = Path("Data/diagnostics")
DIAGNOSTIC_FRAMES = {}  # name -> frame, all written out in the last cell

# Open a fresh read-only connection.
diagnostics_connection = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)

# Thresholds are read back out of QUERY itself, so these checks can't drift.
LIMITS = {name: float(limit) for name, limit in re.findall(r"cp\.(\w+)\s*<=\s*([\d.]+)", QUERY)}
MW_MAX = LIMITS.pop("mw_freebase")

# Rows that clear MW on its own: the denominator for the next two checks.
WITHIN_MW = f"""
    FROM compound_structures AS cs
    JOIN compound_properties AS cp USING (molregno)
    JOIN molecule_dictionary AS md USING (molregno)
    WHERE cs.canonical_smiles IS NOT NULL
      AND cp.mw_freebase <= {MW_MAX:g}
"""
n_within_mw = diagnostics_connection.execute(f"SELECT COUNT(*) {WITHIN_MW}").fetchone()[0]

# Re-read the file so this section stands on its own, and parse once for reuse.
fragments = pd.read_csv(FRAGMENTS)
mols = fragments["smiles"].map(Chem.MolFromSmiles)

print("QUERY filters on: " + ", ".join([f"mw_freebase <= {MW_MAX:g}"]
                                       + [f"{name} <= {limit:g}" for name, limit in LIMITS.items()]))
print(f"rows within MW alone: {n_within_mw:,}")
print(f"fragments on disk:    {len(fragments):,} (unparsable: {int(mols.isna().sum())})")

### 1. Within the MW limit, rejected by something else

A property that is `NULL` fails the comparison in `QUERY` too, so those are counted here separate.

In [ ]:
# Any property over its limit, or missing altogether.
why_rejected = " OR ".join(f"cp.{name} IS NULL OR cp.{name} > {limit:g}" for name, limit in LIMITS.items())
rejected = pd.read_sql_query(f"""SELECT md.chembl_id, cs.canonical_smiles AS smiles, cp.mw_freebase, 
                            {", ".join("cp." + name for name in LIMITS)} {WITHIN_MW} AND ({why_rejected})""", 
                            diagnostics_connection,)

over = pd.DataFrame({name: rejected[name] > limit for name, limit in LIMITS.items()})
missing = pd.DataFrame({name: rejected[name].isna() for name in LIMITS})
culprits = over | missing
rejected["n_reasons"] = culprits.sum(axis=1)
rejected["reasons"] = [" + ".join(culprits.columns[row]) for row in culprits.to_numpy()]
DIAGNOSTIC_FRAMES["rejected_by_property"] = rejected

print(f"{len(rejected):,} of the {n_within_mw:,} rows within MW "
      f"({100 * len(rejected) / n_within_mw:.1f}%) are rejected by another property")

print(f"all four properties NULL (elements, inorganics): {int(missing.all(axis=1).sum()):,}")
display(pd.DataFrame({"over the limit": over.sum(), "missing (NULL)": missing.sum()}))

# Eyeball the discards: lists the reason for discarding below as well. 
sample = rejected.sample(8, random_state=0)
sample_mols = [Chem.MolFromSmiles(smiles) for smiles in sample["smiles"]]
display(Draw.MolsToGridImage([mol for mol in sample_mols if mol is not None], molsPerRow=4, subImgSize=(220, 180),
                             legends=[reasons for mol, reasons in zip(sample_mols, sample["reasons"]) if mol is not None],))

### 2. Stereoisomers that end up as separate rows

`Chem.MolToSmiles` keeps stereochem, and dedup is on that string, so every
stereoisomer ChEMBL registers separately is its own row and its own point in the
UMAP. Stripping stereo from a copy of each molecule shows which rows are only
separated by it.

In [ ]:
def flatten(mol):
    """Canonical SMILES with stereo stripped. Isotope labels survive,
    so isotopologues aren't mistaken for stereoisomers."""
    flat = Chem.Mol(mol)  # RemoveStereochemistry works in place
    Chem.RemoveStereochemistry(flat)
    return Chem.MolToSmiles(flat)

flat_smiles = mols.map(flatten)

# Group the rows that share a skeleton; a group of one is just a molecule.
repeated = flat_smiles.duplicated(keep=False)
# The line below could've been split up, but you know... 
stereo_sets = (fragments[repeated].assign(flat_smiles=flat_smiles[repeated]).groupby("flat_smiles")
               .agg(n_rows=("smiles", "size"), smiles=("smiles", ";".join), chembl_ids=("chembl_id", ";".join)).sort_values("n_rows", ascending=False))
DIAGNOSTIC_FRAMES["stereoisomer_sets"] = stereo_sets

print(f"rows carrying stereochemistry:      {int((flat_smiles != fragments['smiles']).sum()):,}")
print(f"distinct skeletons ignoring stereo: {flat_smiles.nunique():,} of {len(fragments):,} rows")

# The worst offender, drawn out.
largest = stereo_sets.index[0]
members = stereo_sets.loc[largest, "smiles"].split(";")
print(f"largest set ({len(members)} rows): {largest}")
display(Draw.MolsToGridImage([Chem.MolFromSmiles(smiles) for smiles in members], molsPerRow=4, subImgSize=(200, 170),
                             legends=stereo_sets.loc[largest, "chembl_ids"].split(";"),))

### 3. Frags too small to mean anything

The rule have no floor, so methane, water and bare halide ions clear it.<br>
Two overlapping ways of catching them:
1. Three heavy atoms or fewer, and no carbon at all. 
2. Counter-ions that survived desalting because ChEMBL registered them as a separate entry.

In [ ]:
TINY_HEAVY_ATOMS = 3

heavy_atoms = mols.map(lambda mol: mol.GetNumHeavyAtoms())
mol_weight = mols.map(Descriptors.MolWt)
has_carbon = mols.map(lambda mol: any(atom.GetSymbol() == "C" for atom in mol.GetAtoms()))

tiny = (fragments.assign(heavy_atoms=heavy_atoms, mol_weight=mol_weight.round(2), has_carbon=has_carbon).loc[(heavy_atoms <= TINY_HEAVY_ATOMS) | ~has_carbon]
        .sort_values(["heavy_atoms", "mol_weight"]))
DIAGNOSTIC_FRAMES["tiny_fragments"] = tiny

# Visual check of the tiny frags. 
display(Draw.MolsToGridImage([Chem.MolFromSmiles(smiles) for smiles in tiny["smiles"].head(24)], molsPerRow=6, subImgSize=(150, 130),
                             legends=tiny["smiles"].head(24).tolist(),))

### Write the frames out

One CSV per check, in `Data/diagnostics/`. List columns are `;`-joined so they survive the round trip.

In [ ]:
DIAGNOSTICS.mkdir(parents=True, exist_ok=True)

manifest = []
for name, frame_out in DIAGNOSTIC_FRAMES.items():
    path = DIAGNOSTICS / f"{name}.csv"
    # Keep the index only where it carries something (a SMILES, a property name)
    # rather than a row number.
    frame_out.to_csv(path, index=not pd.api.types.is_integer_dtype(frame_out.index))
    manifest.append({"frame": name,
                     "rows": len(frame_out),
                     "columns": frame_out.shape[1],
                     "file": str(path)})

display(pd.DataFrame(manifest).set_index("frame"))

# Get Morgan fprints

Unparsable rows drop out and `kept` maps the matrix back onto the frame.

In [ ]:
frame = pd.read_csv(FRAGMENTS)
generator = rdFingerprintGenerator.GetMorganGenerator(radius=RADIUS, fpSize=N_BITS)

smiles_list = frame["smiles"].tolist()
matrix = np.zeros((len(smiles_list), N_BITS), dtype=np.uint8) # Expects a uint8
kept = []

for i, smi in enumerate(smiles_list):
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        matrix[len(kept)] = generator.GetFingerprintAsNumPy(mol)
        kept.append(i)
    else:
        print(f"  {i:,} {smi} -> None")

matrix = matrix[: len(kept)]
frame = frame.iloc[kept].reset_index(drop=True)
bits = matrix.sum(axis=1)

# Quick sanity checks for dims, emtpy fingerprints, and bits never used
print(f"fingerprint matrix: {matrix.shape} ({matrix.nbytes / 1e6:.0f} MB)")
print("all-zero fingerprints:", int((bits == 0).sum()))
print("bits never used:", int((matrix.sum(axis=0) == 0).sum()), "of", N_BITS)

## Fit UMAP

Fits UMAP, saves the coords, and prints inspection info. 

In [ ]:
import umap 

# This does not include random_state=42, because seeding makes UMAP fall back to
# a single-threaded implementation. 
reducer = umap.UMAP(n_neighbors=N_NEIGHBORS,
                    min_dist=MIN_DIST,
                    n_components=2,
                    metric="jaccard",
                    n_jobs=N_JOBS,
                    #random_state=42,
                    verbose=True,)
embedding = reducer.fit_transform(matrix)

# Save coords and print some info 
print("embedding:", embedding.shape)
frame["umap_1"] = embedding[:, 0]
frame["umap_2"] = embedding[:, 1]
frame.to_csv(EMBEDDING, index=False)
print(EMBEDDING)
frame.head()

## Plots
Both rendered in PNG as well as SVG for resolution. 

### Denisty plot

Binned counts on a log colour scale. UMAP axes carry no units, so none given. 

In [ ]:
# Colour gradients didn't render neatly, so falling back to a simple blue ramp.
BLUE_RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
GREY = "#52514e"

TITLE = (f"ChEMBL fragment space: UMAP of Morgan fingerprints\n"
         f"{len(frame):,} fragments, radius {RADIUS}, {N_BITS} bits, Jaccard metric")

figure, axes = plt.subplots(figsize=(10, 9))
mesh = axes.hexbin(embedding[:, 0], embedding[:, 1], gridsize=200,
                   cmap=LinearSegmentedColormap.from_list("density", BLUE_RAMP),
                   norm=LogNorm(vmin=1), mincnt=1, linewidths=0.0,)

bar = figure.colorbar(mesh, ax=axes, fraction=0.045, pad=0.02)
bar.set_label("fragments per bin (log scale)", color=GREY)

axes.set_title(TITLE, fontsize=13, loc="left", pad=12)
axes.set_xlabel("UMAP 1", color=GREY)
axes.set_ylabel("UMAP 2", color=GREY)
axes.set_xticks([]); axes.set_yticks([])
axes.set_aspect("equal", adjustable="datalim")
figure.tight_layout()

FIGURE_STEM.parent.mkdir(parents=True, exist_ok=True)
for extension in ("svg", "png"):
    figure.savefig(FIGURE_STEM.with_name(f"Density_{FIGURE_STEM.name}.{extension}"),
                   dpi=200, bbox_inches="tight")
plt.show()

### Scatter plot

Rendered large, so individual frags stay visible instead of saturating into blobs. 

In [ ]:
figure, axes = plt.subplots(figsize=(20, 20))
axes.scatter(embedding[:, 0], embedding[:, 1], s=1.1, c="#2a78d6", alpha=0.9, linewidths=0.0)

axes.set_title(TITLE, fontsize=26, loc="left", pad=24)
axes.set_xlabel("UMAP 1", color=GREY, fontsize=20)
axes.set_ylabel("UMAP 2", color=GREY, fontsize=20)
axes.set_xticks([]); axes.set_yticks([])
axes.set_aspect("equal", adjustable="datalim")
figure.tight_layout()

for extension in ("svg", "png"):
    figure.savefig(FIGURE_STEM.with_name(f"Scatter_{FIGURE_STEM.name}.{extension}"),
                   dpi=200, bbox_inches="tight")
plt.show()

### Rare scaffolds

Murcko scaffold per fragment, then the least represented `RARE_FRACTION` of the set: rarest
scaffolds first, taking whole scaffolds until 5% of the frags are accounted
for. Those go on top of the scatter in red.

Two notes: 
1. Around 2/3 of the scaffolds appear exactly once, so the 5% cut lands inside that tie. *Which* singletons fall inside it is arbitrary (deterministic, but arbitrary) -- the red points are a representative sample of the rare tail, not a privileged subset of it. 
2. Fargs with no ring have no Murcko scaffold at all, so they can never be selected; they are counted separately below.

In [ ]:
from rdkit.Chem.Scaffolds import MurckoScaffold

RARE_FRACTION = 0.05
RED = "#d1344a"

# Murcko = ring systems plus the linkers between them, so an acyclic frag comes back as an empty str.
scaffolds = frame["smiles"].map(lambda smiles: MurckoScaffold.MurckoScaffoldSmiles(mol=Chem.MolFromSmiles(smiles))) 
acyclic = scaffolds == ""

# Rarest first, ties broken on the SMILES so the cut is reproducible, then take
# whole scaffolds while they fit inside the budget.
ranked = (scaffolds[~acyclic].value_counts().rename_axis("scaffold").reset_index(name="n_fragments").sort_values(["n_fragments", "scaffold"], ignore_index=True))
ranked["cumulative"] = ranked["n_fragments"].cumsum()
rare_scaffolds = ranked[ranked["cumulative"] <= RARE_FRACTION * len(frame)]

# Map back onto the mols: every fragment sitting on one of those scaffolds.
is_rare = scaffolds.isin(set(rare_scaffolds["scaffold"])).to_numpy()

print(f"{len(ranked):,} distinct scaffolds over {int((~acyclic).sum()):,} fragments "
      f"({int(acyclic.sum()):,} acyclic, no scaffold)")
print(f"singleton scaffolds: {int((ranked['n_fragments'] == 1).sum()):,} "
      f"({100 * (ranked['n_fragments'] == 1).mean():.0f}% of scaffolds)")
print(f"rare set: {len(rare_scaffolds):,} scaffolds -> {is_rare.sum():,} fragments "
      f"({100 * is_rare.mean():.2f}% of the {len(frame):,} plotted), "
      f"up to {rare_scaffolds['n_fragments'].max()} fragments per scaffold")

TITLE_RARE = (f"ChEMBL fragment space: the least represented {100 * RARE_FRACTION:g}% of Murcko scaffolds\n"
              f"{is_rare.sum():,} fragments on {len(rare_scaffolds):,} rare scaffolds, in red")

figure, axes = plt.subplots(figsize=(20, 20))
axes.scatter(embedding[~is_rare, 0], embedding[~is_rare, 1], s=1.0, c="#2a78d6", alpha=0.9, linewidths=0.0, label="fragments on common scaffolds")
# Slightly larger and drawn last, or 5% of the points disappear under the rest.
axes.scatter(embedding[is_rare, 0], embedding[is_rare, 1], s=2.0, c=RED, alpha=0.95, linewidths=0.0, zorder=3, label=f"fragments on the least represented {100 * RARE_FRACTION:g}% of scaffolds")

axes.set_title(TITLE_RARE, fontsize=26, loc="left", pad=24)
axes.set_xlabel("UMAP 1", color=GREY, fontsize=20)
axes.set_ylabel("UMAP 2", color=GREY, fontsize=20)
axes.set_xticks([]); axes.set_yticks([])
axes.set_aspect("equal", adjustable="datalim")
legend = axes.legend(fontsize=18, markerscale=8, loc="upper right", frameon=False)
figure.tight_layout()

for extension in ("svg", "png"):
    figure.savefig(FIGURE_STEM.with_name(f"Rare_scaffolds_{FIGURE_STEM.name}.{extension}"),
                   dpi=200, bbox_inches="tight")
plt.show()

### Clusters within the rare scaffolds

The same `RARE_FRACTION` from the cell above. HDBSCAN runs on the UMAP coordinates of the rare points only, and the largest few clusters get a colour each; everything else rare -- smaller clusters and the points HDBSCAN calls noise -- stays grey, and the rest of the map keeps its blue.

The knobs are at the top of the cell. `MIN_CLUSTER_SIZE` is the main one, and `CLUSTER_SELECTION_METHOD` has 2 options: `"leaf"` cuts the tree into tight, local clusters, while `"eom"` tends to merge most of the rare tail into one blob covering half the map. `rare_clusters` comes back with a row per rare fragment -- molecule, scaffold and cluster number.

**NOTES** 

It's not particularly informative. Small clusters capture a lot of noise and are not "more similar". Large clusters look promising but containt chemically distinct moeities. Leaf clustering seemingly "misses" a part of a larger cluster. EOM generates broad. 

*Things considered:*
1. For each scaffold, generate a generic murcko. Group by generic murcko, select 3 largest groups, overlay as clusters. **Problem:** Clusters at loose queries (450k sets) are very top heavy (benzene captures 80% of scaffolds). At tight queries (80k sets), only 1 cluster produced. 
2. Tanimoto similarity grouping would be the same as grouping by Jaccard, which is what the UMAP does already. 

In [ ]:
from sklearn.cluster import HDBSCAN

MIN_CLUSTER_SIZE = 200          # smallest group HDBSCAN allowed to call a cluster
MIN_SAMPLES = None              # None -> MIN_CLUSTER_SIZE; lower = looser, fewer points called noise
CLUSTER_SELECTION_METHOD = "eom"   # "leaf" = tight local clusters, "eom" = fewer, much larger ones
N_CLUSTERS_SHOWN = 3            
CLUSTER_COLOURS = ["#d1344a", "#2e9e4f", "#c035c0"]  
RARE_GREY = "#54534f"           

rare_points = embedding[is_rare]
clusterer = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES, cluster_selection_method=CLUSTER_SELECTION_METHOD, copy=True).fit(rare_points)

rare_clusters = pd.DataFrame({"chembl_id": frame.loc[is_rare, "chembl_id"].to_numpy(),
                              "smiles": frame.loc[is_rare, "smiles"].to_numpy(),
                              "scaffold": scaffolds[is_rare].to_numpy(),
                              "umap_1": rare_points[:, 0],
                              "umap_2": rare_points[:, 1],
                              "cluster": clusterer.labels_,})          # -1 is HDBSCAN's noise label

# Prominence = how many rare fragments a cluster holds; -1 is never a cluster.
sizes = rare_clusters.loc[rare_clusters["cluster"] >= 0, "cluster"].value_counts()
shown = sizes.head(N_CLUSTERS_SHOWN).index.tolist()
rank = {label: position + 1 for position, label in enumerate(shown)}
rare_clusters["cluster_rank"] = rare_clusters["cluster"].map(rank).astype("Int64")
rare_clusters = rare_clusters.sort_values(["cluster_rank", "cluster"], na_position="last", ignore_index=True)

summary = (rare_clusters[rare_clusters["cluster_rank"].notna()].groupby(["cluster_rank", "cluster"], observed=True)
           .agg(fragments=("smiles", "size"), scaffolds=("scaffold", "nunique"), umap_1=("umap_1", "mean"), umap_2=("umap_2", "mean"))
           .round(2).assign(colour=CLUSTER_COLOURS[:len(shown)]))

print(f"{len(sizes)} clusters over {is_rare.sum():,} rare fragments, "
      f"{int((clusterer.labels_ == -1).sum()):,} left as noise "
      f"({100 * (clusterer.labels_ == -1).mean():.0f}%)")
print(f"showing the largest {len(shown)}: {summary['fragments'].sum():,} fragments "
      f"({100 * summary['fragments'].sum() / is_rare.sum():.0f}% of the rare set)")
display(summary)

figure, axes = plt.subplots(figsize=(20, 20))
# The whole map first, then the rare tail, then the clusters on top of that.
axes.scatter(embedding[~is_rare, 0], embedding[~is_rare, 1], s=1.0, c="#2a78d6", alpha=0.45, linewidths=0.0, label="fragments on common scaffolds")
other_rare = rare_clusters["cluster_rank"].isna().to_numpy()
axes.scatter(rare_clusters.loc[other_rare, "umap_1"], rare_clusters.loc[other_rare, "umap_2"], s=2.0, c=RARE_GREY, alpha=0.9, linewidths=0.0, zorder=3,
             label=f"other rare-scaffold fragments ({int(other_rare.sum()):,})")

for colour, (position, label) in zip(CLUSTER_COLOURS, enumerate(shown, start=1)):
    members = rare_clusters["cluster"] == label
    axes.scatter(rare_clusters.loc[members, "umap_1"], rare_clusters.loc[members, "umap_2"],
                 s=7.0, c=colour, alpha=0.95, linewidths=0.0, zorder=4,
                 label=f"cluster {label} ({int(members.sum()):,} fragments)")

axes.set_title(f"ChEMBL fragment space: where the least represented {100 * RARE_FRACTION:g}% of "
               f"scaffolds cluster\n{len(shown)} largest HDBSCAN clusters of the "
               f"{is_rare.sum():,} rare fragments, min_cluster_size {MIN_CLUSTER_SIZE}, "
               f"{CLUSTER_SELECTION_METHOD} selection",
               fontsize=24, loc="left", pad=24)
axes.set_xlabel("UMAP 1", color=GREY, fontsize=20)
axes.set_ylabel("UMAP 2", color=GREY, fontsize=20)
axes.set_xticks([]); axes.set_yticks([])
axes.set_aspect("equal", adjustable="datalim")
axes.legend(fontsize=18, markerscale=6, loc="upper right", frameon=False)
figure.tight_layout()

for extension in ("svg", "png"):
    figure.savefig(FIGURE_STEM.with_name(f"Rare_clusters{FIGURE_STEM.name}.{extension}"),
                   dpi=200, bbox_inches="tight")
plt.show()

# What the coloured regions actually are, six molecules from each.
examples = rare_clusters.dropna(subset=["cluster_rank"]).groupby("cluster", observed=True).head(6)
display(Draw.MolsToGridImage([Chem.MolFromSmiles(smiles) for smiles in examples["smiles"]], molsPerRow=6, subImgSize=(200, 160),
                             legends=[f"cluster {cluster}: {chembl_id}" for cluster, chembl_id in zip(examples["cluster"], examples["chembl_id"])],))

# Save the rare clusters in case mols later overlaid. 
rare_clusters.to_csv(DIAGNOSTICS.with_name(f"Rare_clusters_{FIGURE_STEM.name}.csv"), index=False)